In [5]:
!pip install -q langchain-groq

In [6]:
from google.colab import userdata
from langchain_groq import ChatGroq

MODEL = "openai/gpt-oss-120b"

llm = ChatGroq(
    groq_api_key=userdata.get("GROQ_API_KEY"),
    model=MODEL,
    temperature=0.2
)

In [7]:
products = [
    {
        "id": 101,
        "name": "Cotton Casual T-Shirt",
        "category": "T-Shirt",
        "color": "Black",
        "price": 599,
        "sizes": ["S", "M", "L", "XL"],
        "stock": 20
    },
    {
        "id": 102,
        "name": "Slim Fit Jeans",
        "category": "Jeans",
        "color": "Blue",
        "price": 1299,
        "sizes": ["M", "L", "XL"],
        "stock": 10
    },
    {
        "id": 103,
        "name": "Hooded Sweatshirt",
        "category": "Hoodie",
        "color": "Grey",
        "price": 999,
        "sizes": ["S", "M", "L"],
        "stock": 0
    }
]

In [8]:
for product in products:
    print(product)

{'id': 101, 'name': 'Cotton Casual T-Shirt', 'category': 'T-Shirt', 'color': 'Black', 'price': 599, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 20}
{'id': 102, 'name': 'Slim Fit Jeans', 'category': 'Jeans', 'color': 'Blue', 'price': 1299, 'sizes': ['M', 'L', 'XL'], 'stock': 10}
{'id': 103, 'name': 'Hooded Sweatshirt', 'category': 'Hoodie', 'color': 'Grey', 'price': 999, 'sizes': ['S', 'M', 'L'], 'stock': 0}


In [10]:
import functools
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print( f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

In [11]:
@log_call
def search_product(product_name):
    for product in products:
        if product_name.lower().strip() in product["name"].lower():
            return product
    return None

In [12]:
search_product("cotton")

   [log] search_product('cotton',) -> {'id': 101, 'name': 'Cotton Casual T-Shirt', 'category': 'T-Shirt', 'color': 'Black', 'price': 599, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 20}


{'id': 101,
 'name': 'Cotton Casual T-Shirt',
 'category': 'T-Shirt',
 'color': 'Black',
 'price': 599,
 'sizes': ['S', 'M', 'L', 'XL'],
 'stock': 20}

In [13]:
def format_product(product):

    if product is None:
        return "Product not found."

    return f"""
Product ID: {product['id']}
Product Name: {product['name']}
Category: {product['category']}
Color: {product['color']}
Price: Rs. {product['price']}
Available Sizes: {', '.join(product['sizes'])}
Stock: {product['stock']}
"""

In [14]:
product = search_product("jeans")
print(format_product(product))

   [log] search_product('jeans',) -> {'id': 102, 'name': 'Slim Fit Jeans', 'category': 'Jeans', 'color': 'Blue', 'price': 1299, 'sizes': ['M', 'L', 'XL'], 'stock': 10}

Product ID: 102
Product Name: Slim Fit Jeans
Category: Jeans
Color: Blue
Price: Rs. 1299
Available Sizes: M, L, XL
Stock: 10



In [15]:
PRODUCT_SYSTEM = """
ROLE:
You are a friendly AI shopping assistant for an online clothing store.
FACTS:
You will receive product information retrieved from our product catalogue.
RULES:
1. Answer using only the supplied product information.
2. Never invent product prices.
3. Never invent available sizes.
4. Never claim a product is in stock if stock is zero.
5. If a product is not found, say it is unavailable in our catalogue.
6. If information is missing, say "I'll check with the store owner."
FORMAT:
Be friendly and concise.
Use simple English.
"""

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage

def ask_product(question, product_name):

    product = search_product(product_name)

    if product is None:
        return "Sorry, this product is not available in our catalogue."

    product_info = format_product(product)

    messages = [
        SystemMessage(content=PRODUCT_SYSTEM),

        HumanMessage(
            content=f"""
Product Information:
{product_info}

Customer Question:
{question}
"""
        )
    ]

    response = llm.invoke(messages)

    return response.content

In [17]:
print(
    ask_product(
        "What is the price of this product?",
        "Cotton Casual T-Shirt"
    )
)

   [log] search_product('Cotton Casual T-Shirt',) -> {'id': 101, 'name': 'Cotton Casual T-Shirt', 'category': 'T-Shirt', 'color': 'Black', 'price': 599, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 20}
The Cotton Casual T‑Shirt costs Rs. 599.


In [18]:
print(
    ask_product(
        "Which sizes are available?",
        "Slim Fit Jeans"
    )
)

   [log] search_product('Slim Fit Jeans',) -> {'id': 102, 'name': 'Slim Fit Jeans', 'category': 'Jeans', 'color': 'Blue', 'price': 1299, 'sizes': ['M', 'L', 'XL'], 'stock': 10}
The Slim Fit Jeans are available in sizes **M, L, and XL**.


In [19]:
@log_call
def check_size(product_name, requested_size):

    product = search_product(product_name)

    if product is None:
        return "Product not found."

    if product["stock"] <= 0:
        return "Sorry, this product is currently out of stock."

    if requested_size.upper() in product["sizes"]:
        return f"Yes, size {requested_size.upper()} is available."

    return f"Sorry, size {requested_size.upper()} is not available."

In [20]:
print(check_size("Cotton Casual T-Shirt", "M"))

print(check_size("Slim Fit Jeans", "S"))

print(check_size("Hooded Sweatshirt", "L"))

   [log] search_product('Cotton Casual T-Shirt',) -> {'id': 101, 'name': 'Cotton Casual T-Shirt', 'category': 'T-Shirt', 'color': 'Black', 'price': 599, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 20}
   [log] check_size('Cotton Casual T-Shirt', 'M') -> 'Yes, size M is available.'
Yes, size M is available.
   [log] search_product('Slim Fit Jeans',) -> {'id': 102, 'name': 'Slim Fit Jeans', 'category': 'Jeans', 'color': 'Blue', 'price': 1299, 'sizes': ['M', 'L', 'XL'], 'stock': 10}
   [log] check_size('Slim Fit Jeans', 'S') -> 'Sorry, size S is not available.'
Sorry, size S is not available.
   [log] search_product('Hooded Sweatshirt',) -> {'id': 103, 'name': 'Hooded Sweatshirt', 'category': 'Hoodie', 'color': 'Grey', 'price': 999, 'sizes': ['S', 'M', 'L'], 'stock': 0}
   [log] check_size('Hooded Sweatshirt', 'L') -> 'Sorry, this product is currently out of stock.'
Sorry, this product is currently out of stock.


In [21]:
while True:

    product_name = input("\nEnter product name: ")

    if product_name.lower().strip() in ("bye", "exit", "quit"):
        print("Thank you for shopping with us!")
        break

    question = input("Ask your question: ")

    print(
        "Bot:",
        ask_product(question, product_name)
    )


Enter product name: cotton casual t-shirt
Ask your question: can i get order of that product in the evening
   [log] search_product('cotton casual t-shirt',) -> {'id': 101, 'name': 'Cotton Casual T-Shirt', 'category': 'T-Shirt', 'color': 'Black', 'price': 599, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 20}
Bot: I’ll check with the store owner.

Enter product name: bye
Thank you for shopping with us!


In [22]:
for product in products:
    print(format_product(product))


Product ID: 101
Product Name: Cotton Casual T-Shirt
Category: T-Shirt
Color: Black
Price: Rs. 599
Available Sizes: S, M, L, XL
Stock: 20


Product ID: 102
Product Name: Slim Fit Jeans
Category: Jeans
Color: Blue
Price: Rs. 1299
Available Sizes: M, L, XL
Stock: 10


Product ID: 103
Product Name: Hooded Sweatshirt
Category: Hoodie
Color: Grey
Price: Rs. 999
Available Sizes: S, M, L
Stock: 0

